# 🌿 Plant Disease Detection — EDA & Model Training

**Dataset**: New Plant Diseases Dataset (PlantVillage) — 87,856 images, 38 classes  
**Model**: ResNet50 with transfer learning + Grad-CAM  
**Framework**: PyTorch

---
## Notebook Sections
1. Setup & Imports
2. Dataset Overview
3. Class Distribution Analysis
4. Sample Image Grids
5. Healthy vs Diseased Comparison
6. Image Dimension Statistics
7. Data Augmentation Preview
8. Channel-wise Mean/Std Verification
9. Model Architecture
10. Training Loop
11. Training Curves
12. Evaluation & Metrics
13. Confusion Matrix
14. Grad-CAM Visualisation

## 1. Setup & Imports

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models
from torchvision.models import ResNet50_Weights
from sklearn.metrics import classification_report, confusion_matrix

# ── Project root (notebook lives in notebooks/, so go up one level)
ROOT = Path("..")
import sys; sys.path.insert(0, str(ROOT))
from src.transforms import get_train_transforms, get_inference_transforms
from src.model import build_model, DEVICE, NUM_CLASSES
from src.gradcam import GradCAM

# ── Paths
DATA_DIR   = ROOT / "data" / "plantvillage"
MODELS_DIR = ROOT / "models"
FIGURES_DIR = ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

# ── Style
plt.style.use("dark_background")
TEAL   = "#00d4aa"
ORANGE = "#f57935"
BLUE   = "#00aaff"
PALETTE = [TEAL, ORANGE, BLUE, "#f5a623", "#9b59b6", "#e74c3c"]
warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
print(f"Data   : {DATA_DIR.resolve()}")

## 2. Dataset Overview

In [ ]:
splits = ["train", "valid", "test"]
stats = {}

for split in splits:
    split_dir = DATA_DIR / split
    class_counts = {}
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
            class_counts[cls_dir.name] = n
    stats[split] = class_counts

class_names = sorted(stats["train"].keys())
print(f"Total classes : {len(class_names)}")
for split in splits:
    total = sum(stats[split].values())
    print(f"{split:>6} images: {total:,}")

# Save class names for model loading
with open(MODELS_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)
print(f"\nClass names saved to {MODELS_DIR / 'class_names.json'}")

## 3. Class Distribution Analysis

In [ ]:
train_counts = [stats["train"][c] for c in class_names]
human_names  = [c.replace("___", "\n").replace("_", " ") for c in class_names]

fig, ax = plt.subplots(figsize=(18, 7))
bars = ax.barh(human_names, train_counts, color=TEAL, alpha=0.85, edgecolor="none")

# Colour healthy bars differently
for i, cls in enumerate(class_names):
    if "healthy" in cls.lower():
        bars[i].set_color(ORANGE)

ax.set_xlabel("Number of Training Images", fontsize=12)
ax.set_title("Training Set — Class Distribution (Teal = Diseased, Orange = Healthy)",
             fontsize=14, pad=15)
ax.axvline(np.mean(train_counts), color="white", linestyle="--", alpha=0.5, label=f"Mean: {np.mean(train_counts):.0f}")
ax.legend(fontsize=10)
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: figures/class_distribution.png")
print(f"\nMin class: {class_names[np.argmin(train_counts)]} ({min(train_counts)} images)")
print(f"Max class: {class_names[np.argmax(train_counts)]} ({max(train_counts)} images)")

In [ ]:
# Plant-wise grouping bar chart
plant_totals = Counter()
for cls in class_names:
    plant = cls.split("___")[0].replace("_", " ").replace(",", "").strip()
    plant_totals[plant] += stats["train"][cls]

plants  = [p for p, _ in plant_totals.most_common()]
totals  = [plant_totals[p] for p in plants]
colors  = [ORANGE if p in ["Tomato", "Apple", "Grape"] else TEAL for p in plants]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(plants, totals, color=colors, alpha=0.85, edgecolor="none")
ax.set_title("Training Images per Plant Species", fontsize=14, pad=12)
ax.set_ylabel("Total Images")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "plant_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/plant_distribution.png")

## 4. Sample Image Grid

In [ ]:
# 4×4 grid — one random image from 16 random classes
sampled_classes = random.sample(class_names, 16)

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle("Sample Images from 16 Random Classes", fontsize=16, y=1.01)

for ax, cls in zip(axes.flat, sampled_classes):
    cls_dir = DATA_DIR / "train" / cls
    imgs    = list(cls_dir.iterdir())
    img_path = random.choice(imgs)
    img = Image.open(img_path).convert("RGB").resize((224, 224))
    ax.imshow(img)
    label = cls.replace("___", "\n").replace("_", " ")
    color = ORANGE if "healthy" in cls.lower() else TEAL
    ax.set_title(label, fontsize=7, color=color)
    ax.axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "sample_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/sample_grid.png")

## 5. Healthy vs Diseased Comparison

In [ ]:
# Pick 5 plants that have both healthy and diseased classes
plant_classes = {}
for cls in class_names:
    plant = cls.split("___")[0]
    plant_classes.setdefault(plant, {"healthy": None, "diseased": []})
    if "healthy" in cls.lower():
        plant_classes[plant]["healthy"] = cls
    else:
        plant_classes[plant]["diseased"].append(cls)

compare_plants = [p for p, v in plant_classes.items() if v["healthy"] and v["diseased"]][:5]

fig, axes = plt.subplots(len(compare_plants), 2, figsize=(8, 4 * len(compare_plants)))
fig.suptitle("Healthy vs Diseased Comparison", fontsize=16, y=1.01)

for row, plant in enumerate(compare_plants):
    for col, kind in enumerate(["healthy", "diseased"]):
        cls  = plant_classes[plant]["healthy"] if kind == "healthy" else plant_classes[plant]["diseased"][0]
        path = random.choice(list((DATA_DIR / "train" / cls).iterdir()))
        img  = Image.open(path).convert("RGB").resize((224, 224))
        axes[row, col].imshow(img)
        color = ORANGE if kind == "healthy" else TEAL
        axes[row, col].set_title(f"{plant.replace('_',' ')} — {kind.title()}", fontsize=9, color=color)
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "healthy_vs_diseased.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/healthy_vs_diseased.png")

## 6. Image Dimension Statistics

In [ ]:
# Sample 500 images across train split
all_imgs = list((DATA_DIR / "train").rglob("*.jpg")) + list((DATA_DIR / "train").rglob("*.JPG"))
sample   = random.sample(all_imgs, min(500, len(all_imgs)))

widths, heights = [], []
for p in tqdm(sample, desc="Reading dimensions"):
    w, h = Image.open(p).size
    widths.append(w); heights.append(h)

print(f"Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}")
print(f"Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}")
print(f"All images 256×256: {all(w == 256 and h == 256 for w,h in zip(widths,heights))}")

## 7. Data Augmentation Preview

In [ ]:
train_tf = get_train_transforms()
infer_tf = get_inference_transforms()

# Pick one sample image
sample_path = random.choice(list((DATA_DIR / "train" / "Tomato___Early_blight").iterdir()))
original    = Image.open(sample_path).convert("RGB")

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("Data Augmentation Preview — Same Image, 9 Different Augmentations", fontsize=13)

# Original
axes[0, 0].imshow(original.resize((224, 224)))
axes[0, 0].set_title("Original", color=ORANGE)
axes[0, 0].axis("off")

# Inference transform (no augmentation)
import torchvision.transforms.functional as TF
infer_img = infer_tf(original).permute(1, 2, 0).numpy()
infer_img = (infer_img - infer_img.min()) / (infer_img.max() - infer_img.min())
axes[1, 0].imshow(infer_img)
axes[1, 0].set_title("Inference (no aug)", color=BLUE)
axes[1, 0].axis("off")

# 8 augmented versions
positions = [(0,1),(0,2),(0,3),(0,4),(1,1),(1,2),(1,3),(1,4)]
for r, c in positions:
    aug = train_tf(original).permute(1, 2, 0).numpy()
    aug = (aug - aug.min()) / (aug.max() - aug.min())
    axes[r, c].imshow(aug)
    axes[r, c].set_title("Augmented", color=TEAL, fontsize=9)
    axes[r, c].axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/augmentation_preview.png")

## 8. Channel-wise Mean/Std Verification

In [ ]:
# Sample 1000 images to estimate dataset mean/std
from torchvision import transforms as T

to_tensor = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
sample_paths = random.sample(
    list((DATA_DIR / "train").rglob("*.jpg")) + list((DATA_DIR / "train").rglob("*.JPG")),
    1000
)

channel_sum  = torch.zeros(3)
channel_sq   = torch.zeros(3)
n_pixels     = 0

for p in tqdm(sample_paths, desc="Computing mean/std"):
    t = to_tensor(Image.open(p).convert("RGB"))   # (3, 224, 224)
    channel_sum += t.sum(dim=(1, 2))
    channel_sq  += (t ** 2).sum(dim=(1, 2))
    n_pixels    += 224 * 224

mean = channel_sum / n_pixels
std  = (channel_sq / n_pixels - mean ** 2).sqrt()

print(f"Dataset mean (R,G,B): {mean.tolist()}")
print(f"Dataset std  (R,G,B): {std.tolist()}")
print()
print("ImageNet mean (expected): [0.485, 0.456, 0.406]")
print("ImageNet std  (expected): [0.229, 0.224, 0.225]")

## 9. Model Architecture

ResNet50 with transfer learning:
- **Layers 1–3**: Frozen (ImageNet features)
- **Layer 4**: Trainable (domain fine-tuning)
- **FC head**: Replaced with `Linear(2048 → 38)`

In [ ]:
model = build_model(num_classes=NUM_CLASSES)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"Total parameters   : {total_params:,}")
print(f"Trainable params   : {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
print(f"Frozen params      : {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")
print(f"\nFC layer           : Linear(2048 → {NUM_CLASSES})")
print(f"Device             : {DEVICE}")

## 10. Training Loop

In [ ]:
# ── DataLoaders ──────────────────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 2

train_dataset = datasets.ImageFolder(DATA_DIR / "train", transform=get_train_transforms())
valid_dataset = datasets.ImageFolder(DATA_DIR / "valid", transform=get_inference_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")

# Verify class ordering matches our saved class_names.json
assert train_dataset.classes == class_names, "Class order mismatch! Re-check class_names.json"
print("Class ordering: OK")

In [ ]:
# ── Optimizer, scheduler, loss ────────────────────────────────────────────────
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(),     "lr": 1e-3},
])
scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1, verbose=True)
criterion  = nn.CrossEntropyLoss()

# ── Training config ────────────────────────────────────────────────────────────
MAX_EPOCHS     = 20
EARLY_STOP     = 5        # patience epochs
CHECKPOINT     = MODELS_DIR / "plant_disease_resnet50.pth"

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
best_val_loss = float("inf")
no_improve    = 0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, desc="Train" if train else "Valid ", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += images.size(0)
    return total_loss / total, correct / total


print("Starting training...")
print(f"Device: {DEVICE} | Max epochs: {MAX_EPOCHS} | Batch size: {BATCH_SIZE}\n")

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss,   val_acc   = run_epoch(valid_loader, train=False)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}% | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}%")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  --> Best model saved (val_loss={best_val_loss:.4f})")
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {EARLY_STOP} epochs)")
            break

print("\nTraining complete!")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Checkpoint  : {CHECKPOINT}")

## 11. Training Curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training Curves", fontsize=15)

ax1.plot(epochs, history["train_loss"], color=TEAL,   label="Train Loss", linewidth=2)
ax1.plot(epochs, history["val_loss"],   color=ORANGE, label="Val Loss",   linewidth=2, linestyle="--")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss"); ax1.legend(); ax1.grid(alpha=0.2)

ax2.plot(epochs, [a*100 for a in history["train_acc"]], color=TEAL,   label="Train Acc", linewidth=2)
ax2.plot(epochs, [a*100 for a in history["val_acc"]],   color=ORANGE, label="Val Acc",   linewidth=2, linestyle="--")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy"); ax2.legend(); ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/training_curves.png")

## 12. Evaluation & Metrics

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True))
model.eval()

test_dataset = datasets.ImageFolder(DATA_DIR / "test", transform=get_inference_transforms())
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        outputs = model(images.to(DEVICE))
        preds   = outputs.argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print()
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
short_names = [c.split("___")[-1].replace("_", " ")[:18] for c in class_names]

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm, annot=False, fmt="d",
    xticklabels=short_names, yticklabels=short_names,
    cmap="YlOrRd", ax=ax, linewidths=0.3
)
ax.set_title("Confusion Matrix — Test Set", fontsize=16, pad=15)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.tick_params(axis="y", rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/confusion_matrix.png")

## 14. Grad-CAM Visualisation

In [ ]:
from src.model import predict, get_class_names

cam = GradCAM(model)

# Pick 6 test images (mix of correct and incorrect predictions)
sample_classes = random.sample(class_names, 6)
fig, axes = plt.subplots(6, 3, figsize=(12, 24))
fig.suptitle("Grad-CAM Visualisations — Model Attention on Leaf Regions", fontsize=14)

for row, cls in enumerate(sample_classes):
    img_path = random.choice(list((DATA_DIR / "test" / cls).iterdir()))
    pil_img  = Image.open(img_path).convert("RGB")

    result  = predict(model, class_names, pil_img)
    class_idx = class_names.index(result["class_raw"])
    heatmap   = cam.generate(result["image_tensor"], class_idx=class_idx)
    overlay   = cam.overlay(pil_img, heatmap)

    axes[row, 0].imshow(pil_img.resize((224, 224)))
    axes[row, 0].set_title("Original", fontsize=8)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(heatmap, cmap="jet")
    axes[row, 1].set_title("Heatmap", fontsize=8)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay)
    correct = result["class_raw"] == cls
    color   = TEAL if correct else ORANGE
    axes[row, 2].set_title(f"Pred: {result['class_name'].split('—')[-1].strip()}\n"
                            f"({result['confidence']*100:.1f}%) {'OK' if correct else 'WRONG'}",
                            fontsize=7, color=color)
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "gradcam_examples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: figures/gradcam_examples.png")